# 资源约束项目调度问题 (RCPSP)

**类别：** 调度

来源： [https://www.hexaly.com/templates/resource-constrained-project-scheduling-problem-rcpsp](https://www.hexaly.com/templates/resource-constrained-project-scheduling-problem-rcpsp)


## 问题

**在 Resource-Constrained Project Scheduling Problem (RCPSP) 中**，一个项目由一组需要调度的任务组成。每个任务都有一个给定的持续时间，且不能被中断。任务之间存在优先级约束：每个任务必须在其所有后继任务开始之前结束。问题涉及一组可再生资源。每个任务对每种资源都有一个给定的资源需求或权重（可能为零），表示该任务在执行过程中消耗的资源量。每种资源都有一个给定的最大容量：它可以同时处理多个任务，但被处理任务的权重之和不能超过该最大容量。目标是找到一个使完工时间（makespan）最小的调度方案，即所有任务处理完成的时间。

	

### 学到的建模原则

- 添加 [interval decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模任务
- 定义 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来建模累积资源约束


## 数据

我们提供的 Resource-Constrained Project Scheduling Problem (RCPSP) 实例遵循 Patterson [[1]](#footnote-1) 格式：

- 第一行：

- 任务数量（包括两个额外的持续时间为 0 的虚拟任务：源和汇）
- 可再生资源的数量
- 第二行：每种资源的最大容量
- 从第三行开始，对于每个任务：

- 任务的持续时间
- 每种资源的资源需求（权重）
- 后继任务的数量
- 每个后继任务的 ID


## 模型

Resource-Constrained Project Scheduling Problem (RCPSP) 的 Hexaly 模型使用 interval decision variables 来表示任务。每个 interval 的长度等于相应任务的持续时间。

然后我们写出优先级约束：每个任务必须在其任何后继任务开始之前结束。

累积资源约束可以表述如下：对于每种资源以及每个时间槽 t，正在处理的任务所消耗的资源量不能超过该资源的容量。为了建模这些约束，我们对每种资源和每个时间槽的所有活跃任务的权重进行求和。我们使用可变参数的 **and** 公式结合 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)，以确保资源容量在任何时刻都被满足。得益于这种可变参数的 **and**，即使时间范围非常大，约束公式仍然紧凑而高效。

需要最小化的完工时间（makespan）是所有任务结束的时间。


## Results

在 RG300 [[2]](#footnote-2) 文献基准（包含 **300 个任务** 的实例）上，Hexaly Optimizer 在 1 分钟运行时间内对 Resource-Constrained Project Scheduling Problem (RCPSP) 达到了 1.9% 的平均最优性差距。我们的 [Resource-Constrained Project Scheduling Problem (RCPSP) benchmark page](https://www.hexaly.com/benchmark/hexaly-vs-or-tools-on-the-resource-constrained-project-scheduling-problem-rcpsp) 展示了 Hexaly Optimizer 在这一具有挑战性的问题上如何超越 Gurobi 和 OR-Tools 等传统通用优化求解器。

[Explore this benchmark](https://www.hexaly.com/benchmark/hexaly-vs-or-tools-on-the-resource-constrained-project-scheduling-problem-rcpsp)

[1] Patterson, J. H.,( 1984), [A comparison of exact approaches for solving the multiple constrained resource, Project Scheduling Problem](https://doi.org/10.1287/mnsc.30.7.854), Management Science, Vol. 30, p854-867

[2] D. Debels & M. Vanhoucke (2007). [A Decomposition-Based Genetic Algorithm for the Resource-Constrained Project-Scheduling Problem](https://doi.org/10.1287/opre.1060.0358). Operations Research 55(3):457-469.


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


# The input files follow the "Patterson" format
def read_instance(filename):
    with open(filename) as f:
        lines = f.readlines()

    first_line = lines[0].split()

    # Number of tasks
    nb_tasks = int(first_line[0])

    # Number of resources
    nb_resources = int(first_line[1])

    # Maximum capacity of each resource
    capacity = [int(lines[1].split()[r]) for r in range(nb_resources)]

    # Duration of each task
    duration = [0 for i in range(nb_tasks)]

    # Resource weight of resource r required for task i
    weight = [[] for i in range(nb_tasks)]

    # Number of successors
    nb_successors = [0 for i in range(nb_tasks)]

    # Successors of each task i
    successors = [[] for i in range(nb_tasks)]

    for i in range(nb_tasks):
        line = lines[i + 2].split()
        duration[i] = int(line[0])
        weight[i] = [int(line[r + 1]) for r in range(nb_resources)]
        nb_successors[i] = int(line[nb_resources + 1])
        successors[i] = [int(line[nb_resources + 2 + s]) - 1 for s in range(nb_successors[i])]

    # Trivial upper bound for the end times of the tasks
    horizon = sum(duration[i] for i in range(nb_tasks))

    return (nb_tasks, nb_resources, capacity, duration, weight, nb_successors, successors, horizon)


def main(instance_file, output_file, time_limit):
    nb_tasks, nb_resources, capacity, duration, weight, nb_successors, successors, horizon = read_instance(
        instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Interval decisions: time range of each task
        tasks = [model.interval(0, horizon) for i in range(nb_tasks)]

        # Task duration constraints
        for i in range(nb_tasks):
            model.constraint(model.length(tasks[i]) == duration[i])

        # Precedence constraints between the tasks
        for i in range(nb_tasks):
            for s in range(nb_successors[i]):
                model.constraint(tasks[i] < tasks[successors[i][s]])

        # Makespan: end of the last task
        makespan = model.max([model.end(tasks[i]) for i in range(nb_tasks)])

        # Cumulative resource constraints
        for r in range(nb_resources):
            capacity_respected = model.lambda_function(
                lambda t: model.sum(weight[i][r] * model.contains(tasks[i], t)
                                    for i in range(nb_tasks))
                <= capacity[r])
            model.constraint(model.and_(model.range(makespan), capacity_respected))

        # Minimize the makespan
        model.minimize(makespan)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - total makespan
        # - for each task, the task id, the start and end times
        #
        if output_file != None:
            with open(output_file, "w") as f:
                print("Solution written in file", output_file)
                f.write(str(makespan.value) + "\n")
                for i in range(nb_tasks):
                    f.write(str(i + 1) + " " + str(tasks[i].value.start()) + " " + str(tasks[i].value.end()))
                    f.write("\n")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python rcpsp.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
